# Conveyer Pipeline — Step-by-Step

End-to-end walkthrough of the conversation clustering pipeline: ingest → embeddings → clustering → analysis → export.

In [1]:
from conveyer import ingest, analysis, clustering, viz
from conveyer.config import PipelineConfig
from conveyer.models import build_embedder, build_llm
import pandas as pd
import os

path = ""
cfg = PipelineConfig(
    data_path=path, cluster_on="question", n_clusters=12,
    embedding_backend="sentence_transformers", first_turn_only=False,
    use_llm_naming=False, out_dir=".output/",
)

## Step 1 — Data Ingest & Feature Engineering

Load raw conversations, then annotate each row with brand-mention counts, recommendation flags, and rule-based intent labels. On synthetic data, intent accuracy is also printed as a sanity check.

In [2]:
df = ingest.ingest(cfg)
df = analysis.add_brand_features(df)
df = analysis.add_intent(df, cfg)
print(f"[ingest] {df.attrs.get('source')} | rows={len(df)}")
print(f"[features] recommendation_rate={df['is_recommendation'].mean():.1%} | "
      f"brands/answer={df['n_brands_answer'].mean():.2f}")
acc = analysis.intent_accuracy(df)
if acc is not None:
    print(f"[intent] rule accuracy vs ground truth (synthetic): {acc:.1%}")

[ingest] SYNTHETIC (1500 rows;  not found) | rows=1500
[features] recommendation_rate=93.3% | brands/answer=2.36
[intent] rule accuracy vs ground truth (synthetic): 98.0%


## Step 2 — Corpus Preparation & Embeddings

Optionally filter to first-turn messages only, then encode each text into a dense vector. If an LLM is configured and `llm_keyphrase_expansion=True`, keyphrases are extracted first to improve embedding quality.

In [3]:
mask = (df[cfg.col_session_pos] == 1) if cfg.first_turn_only else pd.Series(True, index=df.index)
work = df[mask].copy().reset_index(drop=True)
texts = work[cfg.text_column()].tolist()

llm = build_llm(cfg)
embed_texts = texts
if cfg.llm_keyphrase_expansion and llm is not None:
    embed_texts = clustering.llm_keyphrase_expansion(texts, llm)
    print("[llm] keyphrase expansion applied before embedding")

embedder = build_embedder(cfg)
embeddings, emb_name = clustering.embed_corpus(embed_texts, cfg, embedder=embedder)
print(f"[embeddings] {embeddings.shape} via {emb_name} | docs={len(texts)}")

c:\Users\Petya_\MEGAPROJECTS\NIQ\conveyer\.conv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 12/12 [00:31<00:00,  2.64s/it]

[embeddings] (1500, 1024) via sentence-transformers:BAAI/bge-large-en-v1.5 | docs=1500


## Step 3 — Clustering: Method Comparison & Selection

Five algorithms (kmeans, agglomerative, spectral, GMM, HDBSCAN) are scored by silhouette to find the best fit. K-means is also run across a range of k values (sweep) and its stability is measured with ARI across multiple seeds. BERTopic is attempted as an additional reference partition.

In [4]:
niq = work[cfg.col_niq].astype(str).tolist() if cfg.col_niq in work.columns else None
sweep = clustering.kmeans_sweep(embeddings, cfg)
comparison = clustering.compare_methods(embeddings, cfg, niq=niq)
print("[methods]\n" + comparison.to_string(index=False))

n_clusters = cfg.n_clusters
granularity = None
if cfg.llm_select_granularity and llm is not None:
    granularity = clustering.llm_select_granularity(embeddings, texts, cfg, llm)
    n_clusters = granularity["best_k"]
    print(f"[llm] granularity -> n={n_clusters}")

method = clustering.select_best_method(comparison) if cfg.cluster_method == "auto" else cfg.cluster_method
work["cluster_kmeans"] = clustering.run_kmeans(embeddings, n_clusters, cfg)
work["cluster"] = clustering.cluster(method, embeddings, cfg, n_clusters=n_clusters)
stability = clustering.kmeans_stability(embeddings, n_clusters, cfg)

bertopic_labels, _ = clustering.run_bertopic(texts, embeddings, cfg)
if bertopic_labels is not None:
    work["cluster_bertopic"] = bertopic_labels
    print(f"[bertopic] clusters={clustering.n_clusters_found(bertopic_labels)} | "
          f"noise={clustering.noise_fraction(bertopic_labels):.1%}")
else:
    print("[bertopic] unavailable")

print(f"[clustering] method={method} | n={n_clusters} | kmeans stability(ARI)={stability:.3f}")

c:\Users\Petya_\MEGAPROJECTS\NIQ\conveyer\.conv\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


[methods]
       method  n_clusters  noise_frac  silhouette  davies_bouldin  calinski_harabasz  ARI_vs_NIQ  NMI_vs_NIQ
       kmeans          12    0.000000    0.230636        2.459215          68.965107    0.039746    0.102234
          gmm          12    0.000000    0.222707        2.409349          67.385173    0.033982    0.076322
agglomerative          12    0.000000    0.184206        2.192416          47.310543    0.067887    0.207698
     spectral          12    0.000000    0.091248        2.535785          39.246607    0.142340    0.342734
      hdbscan           3    0.051333    0.057936        1.926952          44.156703    0.004959    0.028890
[bertopic] clusters=11 | noise=41.5%
[clustering] method=kmeans | n=12 | kmeans stability(ARI)=0.738


## Step 4 — Analysis & Interpretation

Compute per-cluster statistics, brand amplification rates, top product recommendations, and optional NIQ validation scores. Clusters are named either by keyword extraction or LLM (if configured).

In [5]:
amp = analysis.brand_amplification(df)
summary = analysis.cluster_summary(work, cfg)
niq_scores = analysis.validate_against_niq(work, cfg)
top_reco = analysis.top_recommendations(df)
top_reco_cluster = analysis.top_recommendations_by_cluster(work)
reps = clustering.representative_docs(embeddings, work["cluster"].to_numpy(), texts)

if cfg.use_llm_naming and llm is not None:
    names = analysis.name_clusters_llm(reps, llm)
else:
    names = analysis.name_clusters_keywords(work, texts)
work["cluster_name"] = work["cluster"].map(lambda c: names.get(int(c), {}).get("label", str(c)))

if niq_scores:
    print("[validation vs NIQ]", {k: {m: round(v, 3) for m, v in d.items()} for k, d in niq_scores.items()})

[validation vs NIQ] {'cluster_kmeans': {'ARI': 0.04, 'NMI': 0.102}, 'cluster_bertopic': {'ARI': 0.055, 'NMI': 0.207}}


## Step 5 — Export & Dashboard

Project embeddings to 2D (UMAP/PCA) for visualization, then write 8 CSV files to the output directory. If `build_dashboard=True`, an interactive HTML dashboard is also generated.

In [6]:
os.makedirs(cfg.out_dir, exist_ok=True)
labeled_cols = [cfg.col_question, "cluster", "cluster_kmeans", "cluster_name", "intent",
                "asks_recommendation", "is_recommendation", "n_brands_answer"]
if cfg.col_niq in work.columns:
    labeled_cols.append(cfg.col_niq)
if "cluster_bertopic" in work.columns:
    labeled_cols.append("cluster_bertopic")
labeled = work[labeled_cols]

xy = viz.project_2d(embeddings, cfg.random_state)
proj_cols = [c for c in (cfg.col_question, "cluster", "cluster_name", "intent") if c in work.columns]
projection = work[proj_cols].copy().rename(columns={cfg.col_question: "question"})
projection["x"], projection["y"] = xy[:, 0], xy[:, 1]

outputs = {
    "labeled_first_turn.csv": labeled,
    "cluster_summary.csv": summary,
    "brand_amplification.csv": amp,
    "top_recommendations.csv": top_reco,
    "top_recommendations_by_cluster.csv": top_reco_cluster,
    "kmeans_sweep.csv": sweep,
    "method_comparison.csv": comparison,
    "projection.csv": projection,
}
for fname, frame in outputs.items():
    frame.to_csv(os.path.join(cfg.out_dir, fname), index=False)
print(f"[export] wrote {len(outputs)} files to {os.path.abspath(cfg.out_dir)}")

if cfg.build_dashboard:
    path = viz.build_dashboard(
        {"config": cfg.__dict__, "work": work, "embeddings": embeddings,
         "projection": projection, "summary": summary, "amplification": amp},
        outputs_dir=cfg.out_dir,
    )
    print(f"[dashboard] {os.path.abspath(path)}")

[export] wrote 8 files to c:\Users\Petya_\MEGAPROJECTS\NIQ\conveyer\.output
[dashboard] c:\Users\Petya_\MEGAPROJECTS\NIQ\conveyer\.output\dashboard.html
